# CYP51 genuine molecular cartoon — faster, resumable 50 ns movies

This notebook **uses the already-created full-backbone aligned PDB/DCD pairs in Google Drive**. It does **not** re-download RunPod files, rerun dynamics, or repeat the 50-segment extraction. It shows genuine α-helices/β-sheets/loops, ligand sticks, heme, iron and selected pocket sidechains.

**Improvement over the previous notebook:** 60 trajectory frames **uniformly spanning the entire 50 ns** (rather than the first 60 frames of 200), 720 × 540 resolution, lighter PyMOL settings, and a Drive-backed PNG checkpoint after **every frame**. The MP4 is built from saved frames, so another Colab interruption cannot erase completed rendering. At 8 fps the default movie lasts 7.5 seconds. A second rerun at the same settings skips rendered frames.

If the earlier 200-frame notebook is still running, **click Stop** on that cell before running this one. No need to restart Colab. The original 50 DCD files and the aligned 200-frame sources remain unchanged.

**Performance note:** PyMOL in virtual Xvfb can still be CPU-limited. This workflow reduces work and preserves progress; it cannot guarantee a specific runtime.


In [ ]:
# Cell 1 — Install the minimal requirements; mount existing Google Drive.
!apt-get -qq update >/dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y xvfb libglu1-mesa libgl1 ffmpeg >/dev/null
!pip -q install 'pymol-open-source==3.2.0a0.post5859' 'mdtraj>=1.11' pyvirtualdisplay pillow

from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
OUTPUT_ROOT=Path('/content/drive/MyDrive/CYP51_MD_3D_cartoon_videos')
for label in ['CAL028','CAL015']:
    base=OUTPUT_ROOT/label
    p=base/f'{label}_full_backbone_cartoon.pdb'
    d=base/f'{label}_full_backbone_cartoon_50ns.dcd'
    assert p.is_file() and d.is_file(), (f'{label}: aligned 200-frame full-backbone PDB/DCD not found. '
        'Finish Cell 3 in the previous true-cartoon notebook once, then return here.')
    print(label, 'cached full-backbone aligned source found:',d.stat().st_size//1024//1024,'MiB')


In [ ]:
# Cell 2 — Genuine PyMOL cartoon, 60 evenly-spaced snapshots spanning 50 ns.
# IMPORTANT: if a run is interrupted, rerun this cell; frames already in Drive are skipped.
import os, shutil, subprocess, tempfile
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from pyvirtualdisplay import Display
import pymol2

WIDTH, HEIGHT = 720, 540      # Both even, H.264 compliant
N_MOVIE_FRAMES = 60          # 60 real frames spanning 50 ns (change to 90 for smoother video)
FPS = 8                      # Duration: 60/8 = 7.5 seconds
SYSTEMS_TO_RENDER = ['CAL028','CAL015'] # Can use ['CAL028'] first
SHOW_DYNAMIC_HBONDS = False  # Faster when False; sticks, heme and ligand still move every frame.

POCKET = {
    'CAL028':[118,132,233,307,308,376,377,378,379,380,507,508],
    'CAL015':[118,121,126,131,132,228,233,376,377,508],
}
HBONDS = {
    'CAL028':[('TYR118','OH','O4'),('ILE379','N','O3'),('PHE380','N','O4'),('SER378','N','O3')],
    'CAL015':[('MET508','O','N4')],
}
FONT='/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'

def is_valid_png(filename):
    try:
        if not filename.is_file() or filename.stat().st_size < 1000:return False
        with Image.open(filename) as im:
            return im.size==(WIDTH,HEIGHT) and im.verify() is None
    except Exception:
        return False

def caption_png(filename, name, time_ns):
    im=Image.open(filename).convert('RGB')
    d=ImageDraw.Draw(im,'RGBA')
    d.rounded_rectangle([12,12,337,66], radius=8, fill=(255,255,255,215))
    f1=ImageFont.truetype(FONT,17)
    f2=ImageFont.truetype(FONT,14)
    d.text((23,18),f'{name} | CYP51 molecular dynamics',font=f1,fill=(25,31,36))
    d.text((23,42),f'50 ns production  |  t = {time_ns:.2f} ns',font=f2,fill=(50,57,64))
    im.save(filename,optimize=True)

def make_movie(system):
    folder=OUTPUT_ROOT/system
    pdb=folder/f'{system}_full_backbone_cartoon.pdb'
    dcd=folder/f'{system}_full_backbone_cartoon_50ns.dcd'
    # Frame cache is linked to dimensions, frame count, and visual settings.
    cache=folder/f'fast_cartoon_{N_MOVIE_FRAMES}f_{WIDTH}x{HEIGHT}_hb{int(SHOW_DYNAMIC_HBONDS)}'
    cache.mkdir(parents=True,exist_ok=True)
    index_file=cache/'source_frame_indices.txt'
    # The 200 saved source frames span t ~0.01 to 49.76 ns.
    indices=np.linspace(0,199,N_MOVIE_FRAMES,dtype=int)
    index_file.write_text('\n'.join(str(i) for i in indices)+'\n')
    remaining=[i for i in range(len(indices)) if not is_valid_png(cache/f'frame_{i:04d}.png')]
    print(f'\n{system}: {len(indices)} real frames cover the complete 50 ns; {len(indices)-len(remaining)} cached, {len(remaining)} remaining',flush=True)
    if remaining:
        display=Display(visible=0,size=(WIDTH,HEIGHT),color_depth=24)
        display.start()
        p=pymol2.PyMOL()
        try:
            p.start()
            cmd=p.cmd
            cmd.reinitialize()
            cmd.load(str(pdb),'complex')
            cmd.load_traj(str(dcd),'complex',state=1)
            assert cmd.count_states('complex')==200, 'Expected 200 aligned saved frames'
            cmd.bg_color('white')
            cmd.set('orthoscopic',1)
            cmd.set('depth_cue',0)
            cmd.set('antialias',0)           # largest speed win with software GL
            cmd.set('specular',0.1)
            cmd.set('ray_shadows',0)
            cmd.set('cartoon_fancy_helices',0)
            cmd.set('cartoon_smooth_loops',0)
            cmd.set('cartoon_flat_sheets',1)
            cmd.set('cartoon_transparency',0)
            cmd.set('stick_radius',0.17)
            cmd.set('sphere_scale',0.42)
            cmd.hide('everything','all')
            cmd.select('protein','complex and polymer.protein')
            cmd.select('ligand','complex and resn LIG')
            cmd.select('heme','complex and resn HEM')
            cmd.select('fe','heme and name FE')
            cmd.select('pocket','protein and resi '+'+'.join(map(str,POCKET[system])))
            assert cmd.count_atoms('protein') >= 1800, 'Missing full peptide backbone'
            assert cmd.count_atoms('ligand') >= 25, 'Ligand selection failed'
            cmd.dss('protein')
            # Display secondary structure around active site; protein outside pocket stays stored.
            cmd.frame(1)
            cmd.select('nearby_backbone','byres (protein within 15 of ligand)')
            cmd.show('cartoon','nearby_backbone')
            cmd.color('gray75','protein')
            cmd.show('sticks','pocket and not name N+CA+C+O')
            cmd.color('gray55','pocket and elem C')
            cmd.show('sticks','ligand')
            cmd.set_color('lig_cyan',[0.05,0.64,0.78])
            cmd.set_color('lig_magenta',[0.78,0.25,0.58])
            cmd.color('lig_cyan' if system=='CAL028' else 'lig_magenta','ligand and elem C')
            cmd.color('red','ligand and elem O')
            cmd.color('blue','ligand and elem N')
            cmd.show('sticks','heme and not name FE')
            cmd.color('salmon','heme and elem C')
            cmd.color('red','heme and elem O')
            cmd.color('blue','heme and elem N')
            cmd.show('spheres','fe')
            cmd.color('orange','fe')
            cmd.orient('ligand or heme or pocket')
            cmd.zoom('ligand or heme or pocket',buffer=5)
            fixed_view=cmd.get_view()
            with tempfile.TemporaryDirectory() as tmp:
                temporary_png=Path(tmp)/'render.png'
                for j in remaining:
                    i=int(indices[j])
                    cmd.frame(i+1)
                    cmd.set_view(fixed_view)
                    if SHOW_DYNAMIC_HBONDS:
                        cmd.delete('contacts')
                        for name,prot_atom,lig_atom in HBONDS[system]:
                            residue=int(''.join(x for x in name if x.isdigit()))
                            dsel=f'protein and resi {residue} and name {prot_atom}'
                            asel=f'ligand and name {lig_atom}'
                            if cmd.count_atoms(dsel)==1 and cmd.count_atoms(asel)==1:
                                if cmd.get_distance(dsel,asel,state=i+1)<=3.5:
                                    cmd.distance('contacts',dsel,asel,cutoff=3.5,state=i+1)
                        cmd.hide('labels','contacts')
                        cmd.color('yellow','contacts')
                    cmd.png(str(temporary_png),width=WIDTH,height=HEIGHT,ray=0,quiet=1)
                    time_ns=.01+i*.25
                    caption_png(temporary_png,system,time_ns)
                    destination=cache/f'frame_{j:04d}.png'
                    shutil.copy2(temporary_png,destination)
                    if (j+1)%10==0 or j==remaining[-1]:
                        print(f'  {system}: saved output frame {j+1}/{N_MOVIE_FRAMES}; progress survives Colab restart',flush=True)
        finally:
            try:p.stop()
            finally:display.stop()
    # Render to LOCAL storage first and copy verified MP4 to Drive.
    assert all(is_valid_png(cache/f'frame_{i:04d}.png') for i in range(N_MOVIE_FRAMES)), 'Some cached frames are missing'
    target=folder/f'{system}_CYP51_TRUE_3D_cartoon_FAST_{N_MOVIE_FRAMES}f.mp4'
    with tempfile.TemporaryDirectory() as temp:
        local=Path(temp)/'movie.mp4'
        subprocess.run([
            'ffmpeg','-y','-hide_banner','-loglevel','error',
            '-framerate',str(FPS),'-i',str(cache/'frame_%04d.png'),
            '-vf',f'scale={WIDTH}:{HEIGHT},format=yuv420p',
            '-c:v','libx264','-preset','ultrafast','-crf','22',
            '-movflags','+faststart',str(local)],check=True)
        assert local.stat().st_size>20000, 'MP4 output unexpectedly small'
        shutil.copy2(local,target)
    print(f'{system}: complete MP4 saved: {target} ({target.stat().st_size/1024/1024:.1f} MiB)',flush=True)
    return str(target)

videos={}
for sys in SYSTEMS_TO_RENDER:
    videos[sys]=make_movie(sys)
print('All requested movies finished.')


In [ ]:
# Cell 3 — Play the completed videos in Colab and print their Drive locations.
from IPython.display import display, Video
for system, video in videos.items():
    print(system, video)
    display(Video(video,embed=True,width=720))
